# S6E9 | 0.94677 Ridge Tower + 4 Stable Rules

A compact, reproducible blend for **Playground Series S6E9 — Predicting Electric Vehicle Purchases**.

**Public ROC AUC: 0.94677**

The idea is simple:

1. align three complementary predictions with a rank-to-normal transform;
2. combine them using frozen weights;
3. apply four fold-checked ordering rules;
4. verify the exact submission schema and bytes.

This notebook reports the score of the exact generated file. It does not use test labels, and the public score is not presented as a private-leaderboard guarantee.

## Why rank → probit?

ROC AUC depends on ordering. Two strong models can rank rows similarly while using different probability scales. For each source prediction `p`, I compute the percentile rank and map it to a standard normal score:

`z = Φ⁻¹(rank(p) / (n + 1))`

I then blend the resulting `z` scores. This makes the weights describe ranking signal rather than arbitrary calibration..

In [ ]:
from pathlib import Path
import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statistics import NormalDist

pd.set_option("display.max_colwidth", 80)

ID = "id"
TARGET = "Will_Buy_EV"
EXPECTED_ROWS = 286_571
EXPECTED_SHA256 = "89072beade58aa38d52d4a2efa3552bef9a71e0be20c6c7b59ae6f79e69a923a"

SOURCES = {
    "Generator-aware ridge": ("ridge_submission.csv", 0.444707),
    "EV Grand Prix — later snapshot": ("grand_prix_v9_submission.csv", 0.331541),
    "EV Grand Prix — earlier snapshot": ("grand_prix_v6_submission.csv", 0.223752),
}


def average_rank(values):
    values = np.asarray(values, dtype=np.float64)
    return pd.Series(values).rank(method="average").to_numpy(np.float64)


def probit_rank(values):
    values = np.asarray(values, dtype=np.float64)
    probabilities = average_rank(values) / (len(values) + 1.0)
    normal = NormalDist()
    return np.fromiter((normal.inv_cdf(float(p)) for p in probabilities),
                       dtype=np.float64, count=len(probabilities))


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


print(f"Frozen blend weight sum: {sum(weight for _, weight in SOURCES.values()):.6f}")

## Inputs and provenance

The three prediction snapshots are stored with unique filenames and SHA-256 hashes. This avoids silently changing the result if an upstream notebook later publishes a new version.

- [heuljax — Generator-Aware Ridge Logistic Regression](https://www.kaggle.com/code/heuljax/kps6e09-generator-aware-ridge-logistic-regression)
- [lucifer19 — EV Grand Prix](https://www.kaggle.com/code/lucifer19/ev-grand-prix-48-engine-cpu-pit-stop-blend)

On Kaggle, attach the competition data and the accompanying **S6E9 Ridge Tower source predictions** dataset.

In [ ]:
if Path("/kaggle/input").exists():
    input_root = Path("/kaggle/input")
    competition_dirs = [
        path.parent
        for path in input_root.rglob("sample_submission.csv")
        if (path.parent / "test.csv").exists()
    ]
    if len(competition_dirs) != 1:
        raise RuntimeError(f"Expected one competition input directory, found: {competition_dirs}")
    competition_dir = competition_dirs[0]
    output_dir = Path("/kaggle/working")
else:
    project_root = next(
        root for root in [Path.cwd(), *Path.cwd().parents]
        if (root / "data/public-mirror/test.csv").exists()
    )
    competition_dir = project_root / "data/public-mirror"
    input_root = project_root / "publish/ridge_tower_sources"
    output_dir = project_root / "publish/notebook_run"

sample = pd.read_csv(competition_dir / "sample_submission.csv")
test = pd.read_csv(competition_dir / "test.csv")
ids = sample[ID].to_numpy()
test = test.set_index(ID).reindex(ids).reset_index()

source_paths = {}
for name, (filename, _) in SOURCES.items():
    matches = list(input_root.rglob(filename))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {filename}, found: {matches}")
    source_paths[name] = matches[0]

print("Competition input:", competition_dir)
print("Rows:", len(sample))
for name, path in source_paths.items():
    print(f"{name}: {path}")

In [ ]:
aligned = {}
qa_rows = []
for name, (filename, weight) in SOURCES.items():
    frame = pd.read_csv(source_paths[name])
    if frame.columns.tolist() != [ID, TARGET]:
        raise AssertionError(f"Unexpected columns in {filename}: {frame.columns.tolist()}")
    if len(frame) != EXPECTED_ROWS or not frame[ID].is_unique:
        raise AssertionError(f"Invalid row count or duplicate IDs in {filename}")
    values = frame.set_index(ID)[TARGET].reindex(ids).to_numpy(np.float64)
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise AssertionError(f"Invalid probabilities in {filename}")
    aligned[name] = values
    qa_rows.append({
        "source": name,
        "weight": weight,
        "rows": len(values),
        "min": values.min(),
        "max": values.max(),
        "sha256": sha256(source_paths[name])[:16] + "…",
    })

qa = pd.DataFrame(qa_rows)
display(qa.style.format({"weight": "{:.6f}", "min": "{:.6g}", "max": "{:.6g}"}))

In [ ]:
names = list(aligned)
rank_matrix = np.column_stack([average_rank(aligned[name]) for name in names])
corr = pd.DataFrame(np.corrcoef(rank_matrix, rowvar=False), index=names, columns=names)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(names[::-1], [SOURCES[name][1] for name in names[::-1]], color=["#2979ff", "#00a676", "#ff9f1c"])
axes[0].set_title("Frozen blend weights")
axes[0].set_xlim(0, 0.5)
axes[0].grid(axis="x", alpha=0.2)

image = axes[1].imshow(corr, vmin=0.99, vmax=1.0, cmap="viridis")
axes[1].set_xticks(range(len(names)), ["Ridge", "GP later", "GP earlier"], rotation=25, ha="right")
axes[1].set_yticks(range(len(names)), ["Ridge", "GP later", "GP earlier"])
axes[1].set_title("Spearman rank correlation")
for i in range(len(names)):
    for j in range(len(names)):
        axes[1].text(j, i, f"{corr.iloc[i, j]:.5f}", ha="center", va="center", color="white")
fig.colorbar(image, ax=axes[1], fraction=0.046)
plt.tight_layout()
plt.show()

## Frozen tower

The weights are intentionally explicit. There is no leaderboard-side search in this notebook.

In [ ]:
tower_score = np.zeros(len(ids), dtype=np.float64)
for name, (_, weight) in SOURCES.items():
    tower_score += weight * probit_rank(aligned[name])

print("Tower score range:", float(tower_score.min()), float(tower_score.max()))

## Four ordering rules

The rules move small, predefined groups to the extremes before a final rerank. That preserves a valid continuous submission while expressing strong monotonic structure found during fold-level checks.

In [ ]:
income = test["Annual_Income_USD"].to_numpy(np.float64)
commute = test["Daily_Commute_km"].to_numpy(np.float64)
no_subsidy = test["Subsidy_Available"].astype(str).to_numpy() == "No"
concern_1 = test["Environmental_Concern_Level"].to_numpy() == 1
anxious = np.isin(test["Range_Anxiety_Level"].astype(str).to_numpy(), ["Medium", "High"])

rules = [
    ("Income ≥ 170537 → top", income >= 170537, +1),
    ("31004 ≤ income ≤ 41970 → bottom", (income >= 31004) & (income <= 41970), -1),
    ("Commute ≥ 83 km → bottom", commute >= 83, -1),
    ("Income 30000 + no subsidy + concern/anxiety → bottom",
     (income == 30000) & no_subsidy & (concern_1 | anxious), -1),
]

ordered = average_rank(tower_score) / len(tower_score)
rule_report = []
for label, mask, direction in rules:
    ordered[mask] += 2.0 * direction
    rule_report.append({"rule": label, "rows": int(mask.sum()), "direction": "top" if direction > 0 else "bottom"})

final_prediction = average_rank(ordered) / len(ordered)
display(pd.DataFrame(rule_report))

In [ ]:
submission = pd.DataFrame({ID: ids, TARGET: final_prediction.astype(np.float64)})

assert submission.columns.tolist() == [ID, TARGET]
assert len(submission) == EXPECTED_ROWS
assert submission[ID].is_unique
assert np.array_equal(submission[ID].to_numpy(), sample[ID].to_numpy())
assert np.isfinite(submission[TARGET]).all()
assert submission[TARGET].between(0, 1).all()

output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "submission.csv"
submission.to_csv(output_path, index=False)

round_trip = pd.read_csv(output_path, float_precision="round_trip")
assert round_trip.columns.tolist() == [ID, TARGET]
assert np.array_equal(round_trip[ID].to_numpy(), ids)
assert np.isfinite(round_trip[TARGET]).all()
assert round_trip[TARGET].between(0, 1).all()

digest = sha256(output_path)
print(f"Saved: {output_path}")
print(f"Rows: {len(submission):,}")
print(f"SHA-256: {digest}")
print(f"Matches scored artifact: {digest == EXPECTED_SHA256}")
display(submission.head())

## Result and lessons

The generated file matches the scored artifact byte for byte and received **0.94677 public ROC AUC**.

What mattered:

- transform predictions to a common AUC-oriented scale before weighting;
- retain a genuinely different ridge/logistic view;
- freeze the weights and rules;
- verify ID order, uniqueness, finite ranges and the final file hash.

What did **not** reliably help later: recursive reweighting, subgroup residual corrections, pseudo-labeling, a small embedding network, and additional nonlinear stacking. Several of those ideas looked positive locally but did not transfer to the public leaderboard.

Thanks to **heuljax** for the generator-aware ridge/logistic model and to all contributors whose public work informed the EV Grand Prix ensemble. If you reuse the blend, please keep the attribution and validate it against your own folds.